# CH101 Wonder3D multiview experiment

This notebook is a research-only extension of the CH101 AI pipeline. It uses the pinned Wonder3D repository to generate six consistent RGB/normal views from the approved front reference, tries NeuS first, and falls back to an explicit RGB-foreground voxel surface only when NeuS fails or times out. Every result goes through the existing non-production evaluation gate; it never enables Unity input or Gate B.

The adaptive runner checks the Colab runtime before heavy setup. With a GPU it continues Wonder3D and NeuS; without one it completes the No-GPU validation workstream and stops before CUDA installation. The model, checkpoint terms, CUDA dependencies, and mesh extraction may still fail; every failure is recorded and no failure can promote a candidate.

In [ ]:
from pathlib import Path
import json
import re
import os
import shutil
import signal
import subprocess
import sys
import time

CHARACTER_CODE = 'CH101'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
WONDER3D_REPO_URL = 'https://github.com/xxlong0/Wonder3D.git'
WONDER3D_COMMIT = 'd894f827aa8c2917761a0dad3ab40df74c7a5b24'
WONDER3D_INFERENCE_SCRIPT = 'test_mvdiffusion_seq.py'
WONDER3D_EXPECTED_VIEWS = 6
WONDER3D_MASK_SOURCE = os.environ.get('RE_CAMP_WONDER3D_MASK_SOURCE', 'rgb-foreground')
WONDER3D_FALLBACK_RESOLUTION = int(os.environ.get('RE_CAMP_WONDER3D_FALLBACK_RESOLUTION', '96'))
WONDER3D_FALLBACK_DILATION = int(os.environ.get('RE_CAMP_WONDER3D_FALLBACK_DILATION', '0'))
# Allow the pinned NeuS extractor enough time to finish on a free Kaggle T4.
NEUS_TIMEOUT_SECONDS = int(os.environ.get('RE_CAMP_NEUS_TIMEOUT_SECONDS', '7200'))
RUNTIME_NAME = os.environ.get('RE_CAMP_RUNTIME', '').strip().lower()
if not RUNTIME_NAME:
    RUNTIME_NAME = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/working').is_dir() else 'colab'
CONTENT_ROOT = Path('/kaggle/working' if RUNTIME_NAME == 'kaggle' else '/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
WONDER3D_DIR = CONTENT_ROOT / 'provider-wonder3D'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / 'wonder3d'
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
MULTIVIEW_DIR = OUTPUT_ROOT / 'multiview'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidate'
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
REVIEW_DIR = OUTPUT_ROOT / 'review'
assert CHARACTER_CODE == 'CH101'
print({'provider': 'wonder3D', 'commit': WONDER3D_COMMIT, 'inferenceScript': WONDER3D_INFERENCE_SCRIPT, 'generatedViewCount': WONDER3D_EXPECTED_VIEWS, 'fallbackMaskSource': WONDER3D_MASK_SOURCE, 'unityInputAllowed': False})

In [ ]:
# Reuse a complete six-view output before requesting a new GPU allocation.
REUSE_WONDER3D = os.environ.get('RE_CAMP_REUSE_WONDER3D', '1') != '0'
MULTIVIEW_REPORT = MULTIVIEW_DIR / 'wonder3d-generation-report.json'
REUSE_MULTIVIEW = False
REUSE_CHECK = {'status': 'NOT_REUSABLE', 'reasons': ['REPORT_MISSING']}
def sha256_path(path):
    import hashlib
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if REUSE_WONDER3D and MULTIVIEW_REPORT.is_file():
    existing = json.loads(MULTIVIEW_REPORT.read_text(encoding='utf-8'))
    reasons = []
    generation_status = existing.get('generationStatus', existing.get('status'))
    if generation_status != 'MULTIVIEW_GENERATED': reasons.append('GENERATION_STATUS_NOT_COMPLETE')
    if existing.get('provider') != 'wonder3D': reasons.append('PROVIDER_MISMATCH')
    if existing.get('providerCommit') != WONDER3D_COMMIT or existing.get('providerRepoHead') != WONDER3D_COMMIT: reasons.append('PROVIDER_COMMIT_MISMATCH')
    if existing.get('generatedViewCount') != WONDER3D_EXPECTED_VIEWS or existing.get('generatedAzimuths') != [0, 45, 90, 180, -90, -45]: reasons.append('VIEW_LAYOUT_MISMATCH')
    if existing.get('unityInputAllowed') is not False or existing.get('productionPromotionAllowed') is not False: reasons.append('GATE_UNLOCKED')
    existing_reference_manifest = REFERENCE_DIR / 'reference-views-manifest.json'
    if not existing_reference_manifest.is_file() or existing.get('referenceManifestSha256') != sha256_path(existing_reference_manifest): reasons.append('REFERENCE_MANIFEST_SHA256_MISMATCH')
    generated_files = existing.get('generatedFiles')
    if not isinstance(generated_files, list) or len(generated_files) < WONDER3D_EXPECTED_VIEWS: reasons.append('GENERATED_VIEW_FILES_INCOMPLETE')
    else:
        for relative_name in generated_files:
            generated_path = (MULTIVIEW_DIR / relative_name).resolve()
            try: generated_path.relative_to(MULTIVIEW_DIR.resolve())
            except ValueError: reasons.append('GENERATED_FILE_ESCAPES_OUTPUT'); continue
            if not generated_path.is_file(): reasons.append(f'GENERATED_FILE_MISSING:{relative_name}')
    REUSE_CHECK = {'status': 'REUSABLE' if not reasons else 'NOT_REUSABLE', 'reasons': reasons}
    REUSE_MULTIVIEW = not reasons
print({'reuseRequested': REUSE_WONDER3D, 'reuseMultiview': REUSE_MULTIVIEW, 'reuseCheck': REUSE_CHECK, 'unityInputAllowed': False})

In [ ]:
# Select GPU or No-GPU work before installing Blender/CUDA dependencies.
def bootstrap_run(command):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True)

bootstrap_run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if not (TOOLS_DIR / '.git').is_dir():
    bootstrap_run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
bootstrap_run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
bootstrap_run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
if not (ART_DIR / '.git').is_dir():
    bootstrap_run(['git', 'clone', ART_REPO_URL, ART_DIR])
bootstrap_run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
bootstrap_run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTIVE_REPORT_PATH = OUTPUT_ROOT / 'adaptive-workstream-report.json'
bootstrap_run([sys.executable, TOOLS_DIR / 'scripts' / 'run_adaptive_workstream.py', '--provider', 'wonder3D', '--art-root', ART_DIR, '--output', ADAPTIVE_REPORT_PATH])
adaptive_report = json.loads(ADAPTIVE_REPORT_PATH.read_text(encoding='utf-8'))
GPU_PREFLIGHT = adaptive_report['runtimePreflight']
GPU_COUNT = GPU_PREFLIGHT.get('gpuCount', 0)
GPU_PROBE = 'adaptive-runtime-preflight'
GPU_WORK_ENABLED = adaptive_report['selectedWorkstream'] == 'GPU'
print({'adaptiveStatus': adaptive_report['status'], 'selectedWorkstream': adaptive_report['selectedWorkstream'], 'reuseMultiview': REUSE_MULTIVIEW, 'gpuCount': GPU_COUNT, 'unityInputAllowed': False})
if not GPU_WORK_ENABLED:
    raise RuntimeError('ADAPTIVE_NO_GPU_COMPLETED: maintenance work finished; Wonder3D and NeuS remain blocked until GPU returns')


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
def run(command, **kwargs):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if shutil.which('blender') is None or shutil.which('xvfb-run') is None:
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'])
for repo_url, repo_dir, ref in ((TOOLS_REPO_URL, TOOLS_DIR, TOOLS_REF), (ART_REPO_URL, ART_DIR, ART_COMMIT), (WONDER3D_REPO_URL, WONDER3D_DIR, WONDER3D_COMMIT)):
    if not (repo_dir / '.git').is_dir():
        run(['git', 'clone', repo_url, repo_dir])
    run(['git', '-C', repo_dir, 'fetch', 'origin', ref])
    checkout_ref = f'origin/{ref}' if repo_dir == TOOLS_DIR else ref
    run(['git', '-C', repo_dir, 'checkout', '--detach', checkout_ref])
assert subprocess.check_output(['git', '-C', WONDER3D_DIR, 'rev-parse', 'HEAD'], text=True).strip() == WONDER3D_COMMIT
if not REUSE_MULTIVIEW:
    wonder_requirements = []
    wonder_skipped_optional = {'torch', 'torchvision', 'torchaudio', 'xformers', 'bitsandbytes', 'decord', 'nerfacc'}
    for requirement in (WONDER3D_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines():
        normalized_requirement = requirement.strip()
        if not normalized_requirement or normalized_requirement.startswith('#'):
            continue
        if normalized_requirement.startswith('--') or normalized_requirement.startswith('http'):
            continue
        if normalized_requirement.lower().split('==', 1)[0].strip() in wonder_skipped_optional:
            continue
        if normalized_requirement.lower().split('==', 1)[0].strip() == 'pymcubes':
            normalized_requirement = 'PyMCubes==0.1.6'
        wonder_requirements.append(normalized_requirement)
    print({'skippedProviderTorchPins': True, 'remainingRequirementCount': len(wonder_requirements)})
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', *wonder_requirements])
    # Wonder3D's legacy requirements can leave accelerate with an older Hub.
    # 0.23.4 provides split_torch_state_dict_into_shards while retaining the
    # cached_download compatibility expected by the pinned diffusers release.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'huggingface-hub==0.23.4'])
    # Kaggle's current Transformers 5.x requires a newer Hub API than the
    # pinned Wonder3D stack.  Keep the provider on a Python 3.12 wheel
    # compatible with Hub 0.23.4 and diffusers 0.19.x.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', '--force-reinstall', '--no-deps', 'transformers==4.38.2', 'tokenizers==0.15.2'])
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'onnxruntime==1.20.1'])
    # The pinned diffusers release imports the legacy jax.random.KeyArray type.
    # Kaggle's newer JAX removes it, so keep the provider environment compatible.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'jax==0.4.28', 'jaxlib==0.4.28'])
    # Wonder3D uses the PyTorch path; Kaggle's newer Flax is incompatible
    # with the pinned JAX and is not needed for this provider.
    run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'flax'])
    # rembg imports pymatting's optional CuPy path during startup.
    # Pin a wheel compatible with the Kaggle NumPy/CUDA ABI.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', 'cupy-cuda12x==13.4.0'])
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'fastrlock'])
    # Keep Hub pinned after optional binary packages finish resolving.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'huggingface-hub==0.23.4'])
    # tiny-cuda-nn is optional for the pinned Wonder3D PyTorch path.
    # Source builds can stall on free hosted runners, so opt in explicitly.
    if os.environ.get('RE_CAMP_INSTALL_TINY_CUDA_NN', '0') == '1':
        try:
            run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch'])
            TINY_CUDA_NN_STATUS = 'INSTALLED'
        except subprocess.CalledProcessError as error:
            TINY_CUDA_NN_STATUS = 'BLOCKED_OPTIONAL_BUILD'
            print({'tinyCudaNN': TINY_CUDA_NN_STATUS, 'reason': str(error), 'unityInputAllowed': False})
    else:
        TINY_CUDA_NN_STATUS = 'SKIPPED_OPTIONAL'
        print({'tinyCudaNN': TINY_CUDA_NN_STATUS, 'unityInputAllowed': False})
    # tiny-cuda-nn may resolve an older Hub transitively; restore the
    # provider-compatible version after all optional installs finish.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', 'huggingface-hub==0.23.4'])
else:
    print('REUSED_EXISTING_MULTIVIEW: skip Wonder3D and tiny-cuda-nn installation')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BLENDER_PYTHON_SITE = CONTENT_ROOT / 'blender-python-site'
BLENDER_PYTHON_SITE.mkdir(parents=True, exist_ok=True)
if not (BLENDER_PYTHON_SITE / 'numpy').is_dir():
    run([sys.executable, '-m', 'pip', 'install', '-q', '--target', BLENDER_PYTHON_SITE, '--platform', 'manylinux_2_17_x86_64', '--python-version', '3.10', '--only-binary=:all:', '--no-deps', 'numpy==1.23.5'])
BLENDER_ENVIRONMENT = os.environ.copy()
BLENDER_ENVIRONMENT['PYTHONPATH'] = str(BLENDER_PYTHON_SITE)

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
reference_manifest = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
assert reference_manifest['artCommit'] == ART_COMMIT
assert reference_manifest['unityInputAllowed'] is False
FRONT_IMAGE = Path(reference_manifest['views']['front']['path'])
print({'reference': str(FRONT_IMAGE), 'referenceSha256': reference_manifest['views']['front']['sha256'], 'unityInputAllowed': False})

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
reuse_args = ['--reuse-existing'] if REUSE_WONDER3D else []
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_wonder3d_multiview.py', '--provider-repo', WONDER3D_DIR, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', MULTIVIEW_DIR, '--execute'] + reuse_args)
MULTIVIEW_REPORT = MULTIVIEW_DIR / 'wonder3d-generation-report.json'
multiview_report = json.loads(MULTIVIEW_REPORT.read_text(encoding='utf-8'))
assert multiview_report['status'] in {'MULTIVIEW_GENERATED', 'REUSED'}
assert multiview_report['generationStatus'] == 'MULTIVIEW_GENERATED'
assert multiview_report['actualInference'] is False if multiview_report['status'] == 'REUSED' else True
assert multiview_report['unityInputAllowed'] is False
print(json.dumps(multiview_report, indent=2, ensure_ascii=False))

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
# Try the provider's NeuS extractor first, but make the fallback explicit and review-only.
NEUS_DIR = WONDER3D_DIR / 'NeuS'
if not (NEUS_DIR / 'run.sh').is_file():
    raise FileNotFoundError(NEUS_DIR / 'run.sh')
NEUS_CASE = os.environ.get('RE_CAMP_NEUS_CASE', 'CH101_front')
NEUS_WORKER_COUNT = int(os.environ.get('RE_CAMP_NEUS_WORKERS', '0'))
NEUS_END_ITER = int(os.environ.get('RE_CAMP_NEUS_END_ITER', '1000'))
NEUS_SAVE_FREQ = int(os.environ.get('RE_CAMP_NEUS_SAVE_FREQ', str(NEUS_END_ITER)))
NEUS_VAL_FREQ = int(os.environ.get('RE_CAMP_NEUS_VAL_FREQ', '5000'))
NEUS_VAL_MESH_FREQ = int(os.environ.get('RE_CAMP_NEUS_VAL_MESH_FREQ', str(NEUS_END_ITER)))
NEUS_REPORT_FREQ = int(os.environ.get('RE_CAMP_NEUS_REPORT_FREQ', '100'))
for worker_file in NEUS_DIR.rglob('exp_runner.py'):
    worker_source = worker_file.read_text(encoding='utf-8')
    worker_updated = worker_source.replace('num_workers=64', f'num_workers={NEUS_WORKER_COUNT}')
    if worker_updated != worker_source:
        worker_file.write_text(worker_updated, encoding='utf-8')
conf_path = NEUS_DIR / 'confs' / 'wmask.conf'
conf_text = conf_path.read_text(encoding='utf-8')
for config_key, config_value in {
    'end_iter': NEUS_END_ITER,
    'save_freq': NEUS_SAVE_FREQ,
    'val_freq': NEUS_VAL_FREQ,
    'val_mesh_freq': NEUS_VAL_MESH_FREQ,
    'report_freq': NEUS_REPORT_FREQ,
}.items():
    conf_text = re.sub(r'^\s*' + config_key + r'\s*=.*$', f'{config_key} = {config_value}', conf_text, flags=re.MULTILINE)
conf_path.write_text(conf_text, encoding='utf-8')
NEUS_INPUT_ROOT = MULTIVIEW_DIR / 'neus-input'
NEUS_CASE_DIR = NEUS_INPUT_ROOT / NEUS_CASE
NEUS_CASE_DIR.mkdir(parents=True, exist_ok=True)
view_names = ['front', 'front_right', 'right', 'back', 'left', 'front_left']
rgb_root = MULTIVIEW_DIR / 'cropsize-192-cfg3.0' / 'CH101_front'
normal_root = MULTIVIEW_DIR / 'cropsize-192-cfg1.0' / 'CH101_front'
for view_name in view_names:
    rgb_source = rgb_root / f'rgb_000_{view_name}.png'
    normal_source = normal_root / f'normals_000_{view_name}.png'
    if not rgb_source.is_file() or not normal_source.is_file():
        raise FileNotFoundError(f'NeuS input missing: {rgb_source} / {normal_source}')
    shutil.copy2(rgb_source, NEUS_CASE_DIR / rgb_source.name)
    shutil.copy2(normal_source, NEUS_CASE_DIR / normal_source.name)
assert len(list(NEUS_CASE_DIR.glob('rgb_000_*.png'))) == 6
assert len(list(NEUS_CASE_DIR.glob('normals_000_*.png'))) == 6
NEUS_COMPAT_REPORT = NEUS_DIR / 're-camp-neus-compat-report.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'patch_wonder3d_neus_runtime.py', '--neus-dir', NEUS_DIR, '--output', NEUS_COMPAT_REPORT])
mesh_roots = [NEUS_DIR / 'exp' / 'neus' / NEUS_CASE, NEUS_INPUT_ROOT]
def run_neus_with_process_group_timeout():
    command = ['bash', 'run.sh', NEUS_INPUT_ROOT, NEUS_CASE]
    process = subprocess.Popen(command, cwd=NEUS_DIR, start_new_session=True)
    try:
        return_code = process.wait(timeout=NEUS_TIMEOUT_SECONDS)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=5)
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code
mesh_extraction_status = 'NOT_STARTED'
neus_error = None
mesh_candidates = []
NEUS_RUN_STARTED_AT = time.time()
try:
    print({'meshExtractor': 'NeuS', 'endIter': NEUS_END_ITER, 'timeoutSeconds': NEUS_TIMEOUT_SECONDS, 'unityInputAllowed': False})
    run_neus_with_process_group_timeout()
    mesh_candidates = sorted((path for root in mesh_roots if root.is_dir() for path in root.rglob('*') if path.is_file() and path.suffix.lower() in {'.ply', '.obj', '.glb', '.gltf'}), key=lambda path: path.stat().st_mtime)
    if not mesh_candidates:
        raise RuntimeError('Wonder3D mesh extraction produced no supported mesh file')
    # A successful run may leave multiple validation meshes; the newest one
    # reflects the final iteration rather than a stale early export.
    MESH_PATH = mesh_candidates[-1]
    mesh_extraction_status = 'NEUS_MESH_EXTRACTED'
except Exception as error:
    neus_error = repr(error)
    partial_mesh_candidates = sorted((path for root in mesh_roots if root.is_dir() for path in root.rglob('*') if path.is_file() and path.suffix.lower() in {'.ply', '.obj', '.glb', '.gltf'} and path.stat().st_mtime >= NEUS_RUN_STARTED_AT - 2), key=lambda path: path.stat().st_mtime)
    if partial_mesh_candidates:
        MESH_PATH = partial_mesh_candidates[-1]
        mesh_extraction_status = 'NEUS_MESH_PARTIAL_TIMEOUT'
        print({'meshExtractor': 'NeuS', 'status': mesh_extraction_status, 'error': neus_error, 'mesh': str(MESH_PATH), 'unityInputAllowed': False})
    else:
        mesh_extraction_status = 'EXPERIMENTAL_VOXEL_SURFACE_MESH'
        FALLBACK_DIR = OUTPUT_ROOT / 'voxel-fallback'
        print({'meshExtractor': 'NeuS', 'status': 'BLOCKED_FALLBACK_USED', 'error': neus_error, 'fallbackMaskSource': WONDER3D_MASK_SOURCE, 'unityInputAllowed': False})
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'build_wonder3d_voxel_surface.py', '--rgb-dir', rgb_root, '--normal-dir', normal_root, '--output-dir', FALLBACK_DIR, '--reference-manifest', REFERENCE_MANIFEST, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE, '--resolution', str(WONDER3D_FALLBACK_RESOLUTION), '--dilation', str(WONDER3D_FALLBACK_DILATION), '--mask-source', WONDER3D_MASK_SOURCE])
        MESH_PATH = FALLBACK_DIR / f'{CHARACTER_CODE}_wonder3D_voxel_surface_v001.obj'
        if not MESH_PATH.is_file():
            raise RuntimeError('Wonder3D fallback did not produce an OBJ mesh')
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_wonder3d_candidate.py', '--mesh', MESH_PATH, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', CANDIDATE_DIR])
CANDIDATE_MANIFEST = CANDIDATE_DIR / 'candidate-manifest.json'
candidate_manifest = json.loads(CANDIDATE_MANIFEST.read_text(encoding='utf-8'))
assert candidate_manifest['unityInputAllowed'] is False
print({'mesh': str(MESH_PATH), 'meshExtractionStatus': mesh_extraction_status, 'candidateManifest': str(CANDIDATE_MANIFEST), 'unityInputAllowed': False})

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
entry = candidate_manifest['candidates'][0]
candidate_id = 'wonder3D-CH101-001'
candidate_output = EVALUATION_DIR / candidate_id
candidate_output.mkdir(parents=True, exist_ok=True)
launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
refined_glb = candidate_output / f'{candidate_id}_refined.glb'
refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
refinement_report = candidate_output / 'refinement-report.json'
run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', 'wonder3D', '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve'], env=BLENDER_ENVIRONMENT)
# Blender 3.0 can save the refined Blend while its GLB exporter fails; keep the original transport input as the evaluation fallback.
evaluation_candidate = refined_glb if refined_glb.is_file() else Path(entry['modelPath'])
reuse_refined_blend = refined_blend if not refined_glb.is_file() and refined_blend.is_file() else None
evaluation_report = candidate_output / 'evaluation-report.json'
normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
def build_evaluation_args(candidate_path):
    args = ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py', '--', '--candidate', candidate_path, '--candidate-id', candidate_id, '--output-dir', candidate_output, '--report', evaluation_report, '--normalized-blend', normalized_blend, '--integrity-blend', refined_blend]
    if reuse_refined_blend is not None:
        args.extend(['--reuse-normalized-blend', reuse_refined_blend])
    elif os.environ.get('RE_CAMP_REUSE_NORMALIZED_BLEND', '0') == '1' and normalized_blend.is_file():
        args.extend(['--reuse-normalized-blend', normalized_blend])
    return args
evaluation_args = build_evaluation_args(evaluation_candidate)
run(launcher + evaluation_args, env=BLENDER_ENVIRONMENT)
score_report = candidate_output / 'candidate-score.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--output', score_report])
score_data = json.loads(score_report.read_text(encoding='utf-8'))
if score_data.get('orientationValidation', {}).get('correctionRequired'):
    print(f'Upside-down candidate detected; applying vertical polarity correction: {candidate_id}')
    run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', 'wonder3D', '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve', '--invert-up-axis'], env=BLENDER_ENVIRONMENT)
    run(launcher + build_evaluation_args(evaluation_candidate), env=BLENDER_ENVIRONMENT)
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--output', score_report])
    score_data = json.loads(score_report.read_text(encoding='utf-8'))
    if score_data.get('orientationValidation', {}).get('correctionRequired'):
        raise RuntimeError(f'VERTICAL_POLARITY_CORRECTION_FAILED:{candidate_id}')
RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py', '--output', RANKING_MANIFEST, '--score-report', score_report])
ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
assert ranking['unityInputAllowed'] is False
print(json.dumps(ranking, indent=2, ensure_ascii=False))

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
archive_base = CONTENT_ROOT / f're-camp-{CHARACTER_CODE}-wonder3d-NOT-PRODUCTION'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
print({'archive': str(archive_path), 'status': ranking.get('status'), 'selectedCandidate': ranking.get('selectedCandidate'), 'unityInputAllowed': ranking.get('unityInputAllowed')})
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print('Browser download is unavailable; copy the archive before the session ends.')